In [ ]:
!pip install transformers langchain-huggingface sentence-transformers faiss-cpu torch PyMuPDF gradio --q

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

llm_model_id = "google/flan-t5-large"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    llm_model_id,
    device_map="auto",   # Auto-distributes on CPU/GPU/multi-GPU
    torch_dtype="auto"   # float16/bfloat16 if supported
)

# Model
llm_model = AutoModelForSeq2SeqLM.from_pretrained(llm_model_id)

# Pipeline for inference
generator = pipeline(
    "text2text-generation",
    model=llm_model,
    tokenizer=tokenizer
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
max_input_token_len = tokenizer.model_max_length
print("token max length:", max_input_token_len)

token max length: 512


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

encode_kwargs = {'normalize_embeddings': True}
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs=encode_kwargs
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
# ----------------- Search in FAISS -----------------
def search(question, top_k):
    # Embed the question
    question_embedding = embedding_model.embed_query(question)
    question_embedding = np.array([question_embedding], dtype="float32")

    # FAISS search with Cosine Similarity
    similarity_scores, indices = index.search(question_embedding, top_k)

    results = []
    for sim, idx in zip(similarity_scores[0], indices[0]):
        results.append({"chunk": chunks[idx], "similarity": sim})
    return results


# ----------------- Reranking -----------------
def rerank(question, faiss_results, top_k=3):
    # Prepare pairs (question, chunk)
    pairs = [(question, r["chunk"]) for r in faiss_results]

    # Predict relevance scores with CrossEncoder
    scores = reranker.predict(pairs)

    # Attach scores to results
    for i, r in enumerate(faiss_results):
        r["rerank_score"] = float(scores[i])

    # Sort by rerank score (descending)
    reranked = sorted(faiss_results, key=lambda x: x["rerank_score"], reverse=True)

    return reranked[:top_k]


# ----------------- Prompt Builder -----------------
def build_prompt(chunks, question, tokenizer, max_len):
    """
    Build a prompt that does not exceed max_len tokens.
    Keeps space for the question and instructions.
    Truncates context if needed.
    """
    header = (
        "Answer the question based only on the following context.\n"
        "If the information is not present, say: \"Not found in the document !\"\n\n"
        "Context:\n"
    )
    tail = f"\n\nQuestion: {question}\nAnswer:"

    # Count reserved tokens for header + question
    reserved_tokens = len(tokenizer.tokenize(header + tail))
    budget = max_len - reserved_tokens

    # Add chunks until budget is reached
    selected, used = [], 0
    for ch in chunks:
        t = len(tokenizer.tokenize(ch))
        if used + t > budget:
            break
        selected.append(ch)
        used += t

    return header + "\n".join(selected) + tail


def generate_answer(question, top_k=3, sim_threshold=0.3, tokenizer=tokenizer, generator=generator):
    # Initial retrieval
    retrieved = search(question, top_k=10)

    highest_similarity = retrieved[0]["similarity"]  # FAISS returns sorted results
    if highest_similarity < sim_threshold:
        print(
            f"No relevant info found in document. Highest similarity: {highest_similarity:.3f})"
        )
        return "Not found in the document !"

    # Rerank
    reranked = rerank(question, retrieved, top_k=top_k)

    # Build safe prompt
    context_chunks = [r["chunk"] for r in reranked]
    max_input_token_len = tokenizer.model_max_length
    prompt = build_prompt(context_chunks, question, tokenizer, max_len=max_input_token_len)

    # Debug: token count
    tokens = tokenizer.tokenize(prompt)
    print("Number of tokens sent to model:", len(tokens))

    # Generate answer
    out = generator(
        prompt,
        max_new_tokens=500,
        do_sample=False,  # deterministic output
        no_repeat_ngram_size=3  # avoid repetition of phrases
    )[0]["generated_text"]

    return out



In [ ]:
import re
import fitz  # PyMuPDF

def extract_text_from_pdf(file_path):
    with fitz.open(file_path) as pdf:
        text = "\n".join([page.get_text() for page in pdf])

    # Basic cleaning
    text = text.replace("\xa0", " ").replace("\t", " ")
    text = re.sub(r"-\n", "", text)            # fix split words
    text = re.sub(r"\s{2,}", " ", text)        # multiple spaces

    # Merge artificial line breaks
    # If \n is followed by a lowercase letter, replace with space
    # Keep \n if after . ? ! :
    text = re.sub(r"(?<![\.!?;:])\n(?!\n)", " ", text).strip()

    return text

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)


In [ ]:
import faiss
import numpy as np

def build_faiss_index(chunks, embedding_fn=embedding_model.embed_documents):

    embeddings = embedding_fn(chunks)
    embeddings = np.array(embeddings).astype('float32')

    embed_dim = embeddings.shape[1]

    index = faiss.IndexFlatIP(embed_dim)
    index.add(embeddings)

    print(f"Cosine Similarity created with {index.ntotal} vectors of dimension {index.d}")

    # calcul des normes L2
    norms = np.linalg.norm(embeddings, axis=1)

    print("Min norm:", norms.min())
    print("Max norm:", norms.max())

    return chunks, index


In [ ]:
import gradio as gr

# Global state
pdf_loaded = False

def upload_pdf(pdf_path):
    global pdf_loaded, chunks, index
    pdf_loaded = True

    # Build index
    text = extract_text_from_pdf(pdf_path)
    chunks = splitter.split_text(text)
    chunks, index = build_faiss_index(chunks, embedding_model.embed_documents)

    # Enable question box + clear old inputs
    return [gr.update(interactive=True, value=""), ""]

def reset_pdf():
    global pdf_loaded
    pdf_loaded = False
    # Disable question box + clear question + clear answer
    return [gr.update(interactive=False, value=""), ""]

def ask_question(query):
    if not pdf_loaded:
        return "⚠️ Please upload a PDF first."
    # Replace with your RAG retrieval logic
    answer = generate_answer(query)
    return answer

with gr.Blocks() as demo:
    gr.Markdown("# 💬 PDF Chatbot")

    pdf_input = gr.File(label="Upload a PDF", file_types=[".pdf"], type="filepath")

    question = gr.Textbox(
        label="Ask a question",
        placeholder="Type your question and press Enter",
        interactive=False  # disabled until PDF uploaded
    )
    answer = gr.Textbox(label="🤖 Answer")

    # Upload PDF → enable question, clear question & answer
    pdf_input.upload(upload_pdf, pdf_input, [question, answer])

    # Clear PDF → disable + clear question & answer
    pdf_input.clear(reset_pdf, None, [question, answer])

    # Ask question when pressing Enter
    question.submit(ask_question, inputs=question, outputs=answer)

demo.launch(share=True, debug=True, inline=False)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://40913283970a15416b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Cosine Similarity created with 423 vectors of dimension 384
Min norm: 0.9999999
Max norm: 1.0000001
No relevant info found in document. Highest similarity: 0.273)
Number of tokens sent to model: 289
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://40913283970a15416b.gradio.live
